In [2]:
import pandas as pd
import numpy as np

Load Data

In [3]:

df = pd.read_csv("C:\\Users\\Renu Sharma\\Downloads\\Task 3 and 4_Loan_Data (1).csv")
df = df.sort_values('fico_score').reset_index(drop=True)

print(df.head())
print(f"\nTotal records: {len(df)}")
print(f"FICO range: {df['fico_score'].min()} to {df['fico_score'].max()}")

   customer_id  credit_lines_outstanding  loan_amt_outstanding  \
0      7264776                         1           4457.914800   
1      6901345                         3           5281.352243   
2      2585781                         4           6734.984475   
3      1252008                         5           5176.915602   
4      1337395                         5           4271.314690   

   total_debt_outstanding       income  years_employed  fico_score  default  
0             12233.49501  98913.32028               3         408        0  
1             16411.51801  79905.09892               1         409        1  
2             26384.58439  97668.03091               2         418        1  
3             22990.26543  82417.59227               2         425        1  
4             22756.28103  83475.30929               4         438        1  

Total records: 10000
FICO range: 408 to 850


Dynamic programming bucketing

In [4]:
def bucket_fico_scores(df, n_buckets):
    """
    Uses dynamic programming to find optimal FICO score bucket boundaries
    by maximizing log-likelihood.
    
    n_i = total borrowers in bucket i
    k_i = defaulters in bucket i
    p_i = k_i / n_i  (probability of default in bucket)
    
    LL = sum[ k_i * log(p_i) + (n_i - k_i) * log(1 - p_i) ]
    """

    # Get unique FICO scores and their default rates
    fico_groups = df.groupby('fico_score')['default'].agg(['sum', 'count'])
    fico_groups.columns = ['defaults', 'total']
    fico_groups = fico_groups.reset_index()

    scores  = fico_groups['fico_score'].values
    defaults = fico_groups['defaults'].values
    totals   = fico_groups['total'].values
    m = len(scores)

    # Precompute log-likelihood for any range [i, j]
    def log_likelihood(i, j):
        k = defaults[i:j+1].sum()
        n = totals[i:j+1].sum()
        if n == 0 or k == 0 or k == n:
            return 0
        p = k / n
        return k * np.log(p) + (n - k) * np.log(1 - p)

    # Dynamic programming table
    # dp[i][b] = best log-likelihood using first i scores with b buckets
    dp   = np.full((m + 1, n_buckets + 1), -np.inf)
    split = np.zeros((m + 1, n_buckets + 1), dtype=int)
    dp[0][0] = 0

    for b in range(1, n_buckets + 1):
        for i in range(b, m + 1):
            for j in range(b - 1, i):
                val = dp[j][b-1] + log_likelihood(j, i-1)
                if val > dp[i][b]:
                    dp[i][b] = val
                    split[i][b] = j

    # Backtrack to find boundaries
    boundaries = []
    i = m
    b = n_buckets
    while b > 0:
        j = split[i][b]
        boundaries.append(scores[j])
        i = j
        b -= 1
    boundaries.reverse()

    return boundaries


# ── Run with 5 buckets ───────────────────────────────────────────────────────
n_buckets   = 5
boundaries  = bucket_fico_scores(df, n_buckets)

print(f"Optimal bucket boundaries for {n_buckets} buckets:")
print(boundaries)

Optimal bucket boundaries for 5 buckets:
[np.int64(408), np.int64(521), np.int64(581), np.int64(641), np.int64(697)]


Map FICO scores to ratings

In [5]:
def get_rating(fico_score, boundaries):
    """
    Returns rating 1 to n_buckets.
    Rating 1 = best credit (lowest default risk)
    Rating n = worst credit (highest default risk)
    """
    for i, boundary in enumerate(boundaries):
        if fico_score < boundary:
            return i + 1
    return len(boundaries) + 1

# ── Apply rating to all borrowers ────────────────────────────────────────────
df['rating'] = df['fico_score'].apply(lambda x: get_rating(x, boundaries))

# ── Summary of each bucket ───────────────────────────────────────────────────
summary = df.groupby('rating').agg(
    min_fico    = ('fico_score', 'min'),
    max_fico    = ('fico_score', 'max'),
    total       = ('default', 'count'),
    defaults    = ('default', 'sum')
)
summary['default_rate'] = (summary['defaults'] / summary['total'] * 100).round(2)
print("\nBucket Summary:")
print(summary)


Bucket Summary:
        min_fico  max_fico  total  defaults  default_rate
rating                                                   
2            408       520    301       199         66.11
3            521       580   1407       536         38.10
4            581       640   3438       703         20.45
5            641       696   3197       336         10.51
6            697       850   1657        77          4.65


 Test on a single borrower

In [6]:
test_fico = 650
rating = get_rating(test_fico, boundaries)
print(f"FICO score {test_fico} → Rating {rating}")

FICO score 650 → Rating 5
